<a href="https://colab.research.google.com/github/blakeaustin21481275/21481275-smart-finance-assistant/blob/main/week9_worksheet2_gradio_integration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Worksheet 2: Gradio Integration Workshop

**Name:** Blake Austin **Date:** 8/5/2026

## Learning Objectives
By the end of this worksheet, you will:

- Understand how Gradio components are also objects
- Connect your Transaction and Manager classes to a user interface
- Build a working prototype of your Smart Finance App
- See how different types of objects collaborate in a real application

In [ ]:


## Setup: Install and Import


# Install required packages (run this first)
!pip install gradio pandas

# Import what we need
import gradio as gr
import pandas as pd
from datetime import datetime, date
import json


## Part 1: Gradio Objects Discovery

### Task 1.1: Understanding Gradio as Objects

Before we integrate, let's explore how Gradio itself uses object-oriented design.

Ask AI about this:

> "I'm building a user interface for a business app. When I create components like text input boxes, buttons, and dropdowns, I'm making interactive elements that users can click and type in. How is this similar to the way businesses create standardized forms - like how every customer order form has the same structure but holds different information for each customer?"

**AI's explanation:**

What you’re noticing is basically the same idea at two different levels: **reusable structure with variable data**.

### 1. UI components = reusable building blocks

When you build a text input, button, or dropdown, you’re creating a **standardized interactive component**.

* A text input always behaves the same (you type into it)
* A button always triggers an action (submit, save, etc.)
* A dropdown always lets users choose from options

But each time you use them, the **content changes**:

* “Name” field vs “Email” field
* “Submit Order” button vs “Cancel”
* “Country selector” vs “Payment method”

So you’re separating:

* **Structure (component behavior)**
* **Data (what changes per use)**

---

### 2. Business forms = standardized templates

A customer order form works the same way.

Every order form:

* Has the same layout (customer details, items, total, etc.)
* Uses the same fields every time

But each submission contains different data:

* Customer A orders 2 items
* Customer B orders 10 items
* Same structure, different content

This is exactly like a UI component being reused across different screens.

---

### 3. The key similarity

Both systems rely on **standardization for consistency and scalability**:

| UI Components                     | Business Forms                    |
| --------------------------------- | --------------------------------- |
| Reusable inputs (textbox, button) | Reusable form structure           |
| Same behavior everywhere          | Same layout everywhere            |
| Different props/data each time    | Different customer data each time |

---

### 4. Why this matters

Businesses standardize forms for the same reason developers standardize components:

* **Efficiency** → don’t redesign every form/UI from scratch
* **Consistency** → users always know what to expect
* **Accuracy** → structured inputs reduce mistakes
* **Scalability** → easier to add new forms or features

---

### 5. Simple mental model

Think of it like this:

* UI components = **LEGO bricks**
* Business forms = **completed LEGO builds using the same bricks in patterns**
* Data = **what changes inside the structure**

---

If you want, I can map this directly to React (props/state/forms) or show how companies design “form systems” at scale.


### Task 1.2: Simple Gradio Object Exploration

Let's see Gradio objects in action:

In [ ]:
# Create some basic Gradio objects and see their properties
textbox = gr.Textbox(label="Transaction Description", placeholder="Enter description...")
number_input = gr.Number(label="Amount", value=0.0)
dropdown = gr.Dropdown(label="Category", choices=["Food", "Transport", "Entertainment", "Bills", "Income"])

# Print information about these objects
print(f"Textbox type: {type(textbox)}")
print(f"Number input type: {type(number_input)}")
print(f"Dropdown type: {type(dropdown)}")

Textbox type: <class 'gradio.components.textbox.Textbox'>
Number input type: <class 'gradio.components.number.Number'>
Dropdown type: <class 'gradio.components.dropdown.Dropdown'>


**What did you discover about Gradio components?**

---

## Part 2: Building Your Transaction Class (Review & Enhance)

### Task 2.1: Core Transaction Class

Let's start with the Transaction class you developed in Worksheet 1. If you need help, ask AI:

> "I need to create a template for tracking financial transactions in my app. Each transaction should remember its description, amount (negative for expenses, positive for income), category, and date. The template should also be able to tell me if it's an expense and display itself nicely. How would you design this?"

In [ ]:
# Your Transaction class (from Worksheet 1 or AI-generated):

class Transaction:
    def __init__(self, description, amount, category, date=None):
        # Your implementation here
        pass

    def is_expense(self):
        # Your implementation here
        pass

    def __str__(self):
        # Your implementation here
        pass

# Test your Transaction class:
test_transaction = Transaction("Coffee", -4.50, "Food")
print(test_transaction)
print(f"Is expense: {test_transaction.is_expense()}")

TypeError: __str__ returned non-string (type NoneType)

In [ ]:
from datetime import datetime
from collections import defaultdict

# -----------------------------
# Transaction Class
# -----------------------------
class Transaction:
    def __init__(self, description, amount, category, date=None):
        self.description = description
        self.amount = float(amount)  # +income, -expense
        self.category = category
        self.date = date if date else datetime.now()

    def is_expense(self):
        return self.amount < 0

    def is_income(self):
        return self.amount > 0

    def display_row(self):
        return [
            self.date.strftime("%Y-%m-%d"),
            self.description,
            self.category,
            "EXPENSE" if self.is_expense() else "INCOME",
            f"{self.amount:.2f}"
        ]


# -----------------------------
# Transaction Manager
# -----------------------------
class TransactionManager:
    def __init__(self):
        self.transactions = []

    def add_transaction(self, transaction):
        self.transactions.append(transaction)

    # Total calculations
    def total_income(self):
        return sum(t.amount for t in self.transactions if t.is_income())

    def total_expenses(self):
        return sum(t.amount for t in self.transactions if t.is_expense())

    def net_balance(self):
        return sum(t.amount for t in self.transactions)

    # Group by category
    def spending_by_category(self):
        category_totals = defaultdict(float)
        for t in self.transactions:
            category_totals[t.category] += t.amount
        return category_totals

    # Find biggest expenses
    def top_expenses(self, n=3):
        expenses = [t for t in self.transactions if t.is_expense()]
        expenses.sort(key=lambda x: x.amount)  # most negative first
        return expenses[:n]

    # Table display
    def print_table(self):
        headers = ["Date", "Description", "Category", "Type", "Amount"]

        print("\n" + "-" * 80)
        print(f"{headers[0]:<12} {headers[1]:<20} {headers[2]:<15} {headers[3]:<10} {headers[4]:>10}")
        print("-" * 80)

        for t in self.transactions:
            row = t.display_row()
            print(f"{row[0]:<12} {row[1]:<20} {row[2]:<15} {row[3]:<10} {row[4]:>10}")

        print("-" * 80)


# -----------------------------
# Example Usage
# -----------------------------
if __name__ == "__main__":
    manager = TransactionManager()

    # Random sample transactions
    manager.add_transaction(Transaction("Coffee", -6.50, "Food"))
    manager.add_transaction(Transaction("Groceries", -82.30, "Food"))
    manager.add_transaction(Transaction("Salary", 3200.00, "Income"))
    manager.add_transaction(Transaction("Internet Bill", -75.00, "Bills"))
    manager.add_transaction(Transaction("Freelance Work", 450.00, "Income"))
    manager.add_transaction(Transaction("Movie Ticket", -18.00, "Entertainment"))

    # Print table
    manager.print_table()

    # Summary
    print("\nSUMMARY")
    print(f"Total Income:   ${manager.total_income():.2f}")
    print(f"Total Expenses:  ${manager.total_expenses():.2f}")
    print(f"Net Balance:     ${manager.net_balance():.2f}")

    # Category breakdown
    print("\nSPENDING BY CATEGORY")
    for category, total in manager.spending_by_category().items():
        print(f"{category:<15} {total:.2f}")

    # Top expenses
    print("\nTOP EXPENSES")
    for t in manager.top_expenses():
        print(f"{t.description:<20} {t.amount:.2f}")


--------------------------------------------------------------------------------
Date         Description          Category        Type           Amount
--------------------------------------------------------------------------------
2026-05-08   Coffee               Food            EXPENSE         -6.50
2026-05-08   Groceries            Food            EXPENSE        -82.30
2026-05-08   Salary               Income          INCOME        3200.00
2026-05-08   Internet Bill        Bills           EXPENSE        -75.00
2026-05-08   Freelance Work       Income          INCOME         450.00
2026-05-08   Movie Ticket         Entertainment   EXPENSE        -18.00
--------------------------------------------------------------------------------

SUMMARY
Total Income:   $3650.00
Total Expenses:  $-181.80
Net Balance:     $3468.20

SPENDING BY CATEGORY
Food            -88.80
Income          3650.00
Bills           -75.00
Entertainment   -18.00

TOP EXPENSES
Groceries            -82.30
Internet 



---

## Part 3: Simple Finance Manager System

### Task 3.1: Building the Manager Class

Now create a system to manage multiple transactions:

```python
class SimpleFinanceManager:
    def __init__(self):
        self.transactions = []
    
    def add_transaction(self, transaction):
        """Add a Transaction object to our collection"""
        # Your implementation
        pass
    
    def get_total_expenses(self):
        """Calculate total of all expenses (negative amounts)"""
        # Your implementation
        pass
    
    def get_spending_by_category(self, category):
        """Get total spending for a specific category"""
        # Your implementation
        pass
    
    def get_recent_transactions(self, count=5):
        """Get the last N transactions"""
        # Your implementation
        pass
    
    def get_summary(self):
        """Get a summary of all transactions"""
        # Your implementation
        pass

# Test your manager:
manager = SimpleFinanceManager()
# Add some test transactions and verify it works
```


In [ ]:
from collections import defaultdict
from datetime import datetime

class SimpleFinanceManager:
    def __init__(self):
        self.transactions = []

    def add_transaction(self, transaction):
        """Add a Transaction object to our collection"""
        self.transactions.append(transaction)

    def get_total_expenses(self):
        """Calculate total of all expenses (negative amounts)"""
        return sum(t.amount for t in self.transactions if t.amount < 0)

    def get_total_income(self):
        """(Helpful extra) total income"""
        return sum(t.amount for t in self.transactions if t.amount > 0)

    def get_spending_by_category(self, category):
        """Get total spending for a specific category"""
        return sum(
            t.amount for t in self.transactions
            if t.category.lower() == category.lower()
        )

    def get_recent_transactions(self, count=5):
        """Get the last N transactions"""
        return self.transactions[-count:]

    def get_summary(self):
        """Get a summary of all transactions"""
        total_income = self.get_total_income()
        total_expenses = self.get_total_expenses()
        net = total_income + total_expenses

        # Category breakdown
        category_totals = defaultdict(float)
        for t in self.transactions:
            category_totals[t.category] += t.amount

        return {
            "total_transactions": len(self.transactions),
            "total_income": total_income,
            "total_expenses": total_expenses,
            "net_balance": net,
            "category_breakdown": dict(category_totals)
        }


# -----------------------------
# Transaction class (required dependency)
# -----------------------------
class Transaction:
    def __init__(self, description, amount, category, date=None):
        self.description = description
        self.amount = float(amount)
        self.category = category
        self.date = date if date else datetime.now()

    def __repr__(self):
        return f"{self.date.date()} | {self.description} | {self.category} | {self.amount}"


# -----------------------------
# TESTING THE MANAGER
# -----------------------------
manager = SimpleFinanceManager()

# Add sample transactions
manager.add_transaction(Transaction("Coffee", -6.50, "Food"))
manager.add_transaction(Transaction("Groceries", -82.30, "Food"))
manager.add_transaction(Transaction("Salary", 3200.00, "Income"))
manager.add_transaction(Transaction("Internet Bill", -75.00, "Bills"))
manager.add_transaction(Transaction("Freelance Work", 450.00, "Income"))
manager.add_transaction(Transaction("Movie Ticket", -18.00, "Entertainment"))

# -----------------------------
# OUTPUT TESTS
# -----------------------------

print("TOTAL EXPENSES:")
print(manager.get_total_expenses())

print("\nFOOD SPENDING:")
print(manager.get_spending_by_category("Food"))

print("\nRECENT TRANSACTIONS:")
for t in manager.get_recent_transactions(3):
    print(t)

print("\nSUMMARY:")
summary = manager.get_summary()
for key, value in summary.items():
    print(f"{key}: {value}")

TOTAL EXPENSES:
-181.8

FOOD SPENDING:
-88.8

RECENT TRANSACTIONS:
2026-05-08 | Internet Bill | Bills | -75.0
2026-05-08 | Freelance Work | Income | 450.0
2026-05-08 | Movie Ticket | Entertainment | -18.0

SUMMARY:
total_transactions: 6
total_income: 3650.0
total_expenses: -181.8
net_balance: 3468.2
category_breakdown: {'Food': -88.8, 'Income': 3650.0, 'Bills': -75.0, 'Entertainment': -18.0}


---

## Part 4: Connecting Objects to Gradio Interface

### Task 4.1: The Connection Function

This is where the magic happens - your objects work with Gradio objects. Ask AI for help:

> "I have a transaction template and a transaction manager system. I want to create a web form where users can enter transaction details, and when they click 'Add', it should create a new transaction and add it to my manager, then show a confirmation message. How do I connect my business logic to a user interface?"

In [ ]:
# Create a global manager instance
app_manager = SimpleFinanceManager()

def add_transaction_via_gradio(description, amount, category):
    """This function connects Gradio inputs to your objects"""
    try:
        # Create a Transaction object from Gradio inputs
        # Add it to the manager
        # Return a confirmation message
        pass
    except Exception as e:
        return f"Error: {str(e)}"

def get_spending_summary():
    """Get spending summary from the manager"""
    # Your implementation
    pass

# Test the function manually first:
result = add_transaction_via_gradio("Test Coffee", -4.50, "Food")
print(result)

None


In [ ]:
import gradio as gr
demo.launch(share=True)
from datetime import datetime


# -----------------------------
# Transaction Model
# -----------------------------
class Transaction:
    def __init__(self, description, amount, category):
        self.description = description
        self.amount = float(amount)
        self.category = category
        self.date = datetime.now()

    def __repr__(self):
        return f"{self.date.date()} | {self.description} | {self.category} | ${self.amount:.2f}"


# -----------------------------
# Finance Manager
# -----------------------------
class SimpleFinanceManager:
    def __init__(self):
        self.transactions = []

    def add_transaction(self, transaction):
        self.transactions.append(transaction)

    def total_spending(self):
        return sum(t.amount for t in self.transactions)

    def by_category(self):
        summary = {}
        for t in self.transactions:
            summary[t.category] = summary.get(t.category, 0) + t.amount
        return summary

    def biggest_expense(self):
        expenses = [t for t in self.transactions if t.amount < 0]
        if not expenses:
            return None
        return min(expenses, key=lambda t: t.amount)


# Global manager instance
app_manager = SimpleFinanceManager()


# -----------------------------
# UI Bridge Functions
# -----------------------------
def add_transaction_via_gradio(description, amount, category):
    """Connect UI inputs to business logic"""
    try:
        # 1. Create transaction object
        transaction = Transaction(description, amount, category)

        # 2. Add to manager
        app_manager.add_transaction(transaction)

        # 3. Return confirmation
        return f"✅ Added: {transaction}"

    except Exception as e:
        return f"❌ Error: {str(e)}"


def get_spending_summary():
    """Return formatted summary for UI"""
    try:
        total = app_manager.total_spending()
        categories = app_manager.by_category()
        biggest = app_manager.biggest_expense()

        summary_text = f"💰 Total Spending: ${total:.2f}\n\n"

        summary_text += "📊 By Category:\n"
        for cat, amt in categories.items():
            summary_text += f" - {cat}: ${amt:.2f}\n"

        if biggest:
            summary_text += f"\n🚨 Biggest Expense: {biggest.description} (${biggest.amount:.2f})"
        else:
            summary_text += "\n🚨 No expenses yet"

        return summary_text

    except Exception as e:
        return f"❌ Error: {str(e)}"


# -----------------------------
# Gradio UI
# -----------------------------
with gr.Blocks(title="Finance Manager") as app:

    gr.Markdown("# 💼 Simple Finance Manager")

    with gr.Row():
        desc = gr.Textbox(label="Description")
        amount = gr.Number(label="Amount (use negative for expenses)")
        category = gr.Textbox(label="Category")

    add_btn = gr.Button("Add Transaction")
    output = gr.Textbox(label="Result")

    add_btn.click(
        fn=add_transaction_via_gradio,
        inputs=[desc, amount, category],
        outputs=output
    )

    summary_btn = gr.Button("Show Summary")
    summary_output = gr.Textbox(label="Spending Summary")

    summary_btn.click(
        fn=get_spending_summary,
        outputs=summary_output
    )


# Run app
app.launch()

Rerunning server... use `close()` to stop if you need to change `launch()` parameters.
----
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c4bd1dc1c4c433397d.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


KeyboardInterrupt: 

### Task 4.2: Building the Interface

Now create the actual Gradio interface:

In [ ]:
# Create the Gradio interface
with gr.Blocks(title="Smart Finance App Prototype") as demo:

    gr.Markdown("# 💰 Smart Finance App")
    gr.Markdown("Add transactions and see your spending patterns!")

    with gr.Row():
        with gr.Column():
            gr.Markdown("## Add New Transaction")

            # Create Gradio input objects
            desc_input = gr.Textbox(label="Description", placeholder="e.g., Starbucks Coffee")
            amount_input = gr.Number(label="Amount", value=0.0, info="Negative for expenses, positive for income")
            category_input = gr.Dropdown(label="Category",
                                       choices=["Food", "Transport", "Entertainment", "Bills", "Income", "Other"])

            add_button = gr.Button("Add Transaction", variant="primary")

        with gr.Column():
            gr.Markdown("## Summary")
            summary_display = gr.Textbox(label="Current Summary", interactive=False)
            refresh_button = gr.Button("Refresh Summary")

    # Status area
    status_output = gr.Textbox(label="Status", interactive=False)

    # Connect the objects: Button objects call functions that use your custom objects
    add_button.click(
        fn=add_transaction_via_gradio,
        inputs=[desc_input, amount_input, category_input],
        outputs=status_output
    )

    refresh_button.click(
        fn=get_spending_summary,
        outputs=summary_display
    )

# Launch the app
demo.launch(debug=True)


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://eed027f19b182a6d50.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://0b47e37ec55cdfbee1.gradio.live
Killing tunnel 127.0.0.1:7861 <> https://d0b21fa340b03da73a.gradio.live
Killing tunnel 127.0.0.1:7862 <> https://9acb64c029b2abeb45.gradio.live
Killing tunnel 127.0.0.1:7863 <> https://eed027f19b182a6d50.gradio.live




---

## Part 5: Testing Object Collaboration

### Task 5.1: Integration Testing

Test your app by adding several transactions and observing how the objects work together:

**Add these test transactions:**
1. Coffee Shop: -$4.50, Food
2. Bus Ticket: -$3.20, Transport  
3. Salary: +$500.00, Income
4. Groceries: -$67.80, Food

**Document what happens:**

**1. Object Creation:** When you click "Add Transaction", trace what objects get created:

**2. Object Interaction:** How do the Gradio objects pass data to your Transaction objects?

**3. System Updates:** How does the SimpleFinanceManager coordinate everything?

---

## Part 6: Advanced Integration Challenge

### Task 6.1: CSV Loading Feature

Let's add the ability to load transactions from CSV files. Ask AI:

> "I want to add a file upload feature to my finance app so users can load their bank transaction data from CSV files. The CSV has columns for description, amount, category, and date. How do I take this spreadsheet data and integrate it with my existing transaction management system?"


In [1]:
def load_transactions_from_csv(csv_file):
    """Load transactions from uploaded CSV file"""
    if csv_file is None:
        return "No file uploaded"

    try:
        # Your implementation here
        # Read CSV, create Transaction objects, add to manager
        pass
    except Exception as e:
        return f"Error loading CSV: {str(e)}"

# Add this to your Gradio interface (create a new version):
# Include a gr.File() component and connect it to your function

In [2]:
import gradio as gr
import pandas as pd


# ----------------------------
# Transaction Manager
# ----------------------------
class SimpleFinanceManager:
    def __init__(self):
        self.transactions = []

    def add_transaction(self, description, amount, category, date):
        transaction = {
            "description": description,
            "amount": float(amount),
            "category": category,
            "date": date
        }
        self.transactions.append(transaction)

    def get_summary(self):
        if not self.transactions:
            return "No transactions yet."

        total = sum(t["amount"] for t in self.transactions)
        expenses = sum(t["amount"] for t in self.transactions if t["amount"] < 0)
        income = sum(t["amount"] for t in self.transactions if t["amount"] > 0)

        return f"""
Total Transactions: {len(self.transactions)}
Income: ${income:.2f}
Expenses: ${expenses:.2f}
Net Balance: ${total:.2f}
"""


# Create manager instance
manager = SimpleFinanceManager()


# ----------------------------
# Manual transaction entry
# ----------------------------
def add_transaction_via_gradio(description, amount, category):
    manager.add_transaction(description, amount, category, "Manual Entry")
    return f"Added: {description} (${amount})"


def get_spending_summary():
    return manager.get_summary()


# ----------------------------
# CSV Upload Logic
# ----------------------------
def load_transactions_from_csv(csv_file):
    """
    Load transactions from uploaded CSV file.
    CSV must contain:
    description, amount, category, date
    """
    if csv_file is None:
        return "No file uploaded."

    try:
        df = pd.read_csv(csv_file.name)

        required_columns = ["description", "amount", "category", "date"]

        # Check CSV format
        if not all(col in df.columns for col in required_columns):
            return f"CSV missing required columns: {required_columns}"

        # Add each row to manager
        for _, row in df.iterrows():
            manager.add_transaction(
                row["description"],
                row["amount"],
                row["category"],
                row["date"]
            )

        return f"Successfully imported {len(df)} transactions."

    except Exception as e:
        return f"Error loading CSV: {str(e)}"


# ----------------------------
# Gradio Interface
# ----------------------------
with gr.Blocks(title="Smart Finance App Prototype") as demo:

    gr.Markdown("# 💰 Smart Finance App")
    gr.Markdown("Add transactions manually or upload a CSV file.")

    with gr.Row():

        # Left column
        with gr.Column():
            gr.Markdown("## Add New Transaction")

            desc_input = gr.Textbox(
                label="Description",
                placeholder="e.g., Starbucks Coffee"
            )

            amount_input = gr.Number(
                label="Amount",
                value=0.0,
                info="Negative for expenses, positive for income"
            )

            category_input = gr.Dropdown(
                label="Category",
                choices=[
                    "Food",
                    "Transport",
                    "Entertainment",
                    "Bills",
                    "Income",
                    "Other"
                ]
            )

            add_button = gr.Button("Add Transaction", variant="primary")

            gr.Markdown("## Upload CSV")
            csv_upload = gr.File(label="Upload Bank CSV", file_types=[".csv"])
            upload_button = gr.Button("Import CSV")

        # Right column
        with gr.Column():
            gr.Markdown("## Summary")
            summary_display = gr.Textbox(
                label="Current Summary",
                interactive=False,
                lines=8
            )
            refresh_button = gr.Button("Refresh Summary")

    status_output = gr.Textbox(label="Status", interactive=False)

    # Manual add transaction
    add_button.click(
        fn=add_transaction_via_gradio,
        inputs=[desc_input, amount_input, category_input],
        outputs=status_output
    )

    # CSV import
    upload_button.click(
        fn=load_transactions_from_csv,
        inputs=csv_upload,
        outputs=status_output
    )

    # Refresh summary
    refresh_button.click(
        fn=get_spending_summary,
        outputs=summary_display
    )

demo.launch(debug=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://d66c3bf21b31a46e10.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://d66c3bf21b31a46e10.gradio.live


### Task 6.2: Enhanced Interface

Create an enhanced version of your interface that includes CSV loading:

In [ ]:
# Enhanced interface with CSV loading
with gr.Blocks(title="Smart Finance App v2") as enhanced_demo:

    gr.Markdown("# 💰 Smart Finance App v2")

    with gr.Tab("Add Transactions"):
        # Your manual transaction entry interface
        pass

    with gr.Tab("Load from CSV"):
        # Your CSV loading interface
        pass

    with gr.Tab("Analysis"):
        # Your summary and analysis interface
        pass

# Test the enhanced version
enhanced_demo.launch(debug=True)

In [3]:
import gradio as gr
import pandas as pd


# -----------------------------------
# Finance Manager
# -----------------------------------
class SimpleFinanceManager:
    def __init__(self):
        self.transactions = []

    def add_transaction(self, description, amount, category, date):
        transaction = {
            "description": description,
            "amount": float(amount),
            "category": category,
            "date": date
        }
        self.transactions.append(transaction)

    def get_summary(self):
        if not self.transactions:
            return "No transactions loaded."

        total = sum(t["amount"] for t in self.transactions)
        income = sum(t["amount"] for t in self.transactions if t["amount"] > 0)
        expenses = sum(t["amount"] for t in self.transactions if t["amount"] < 0)

        category_totals = {}
        for t in self.transactions:
            cat = t["category"]
            category_totals[cat] = category_totals.get(cat, 0) + t["amount"]

        summary = f"""
Total Transactions: {len(self.transactions)}
Income: ${income:.2f}
Expenses: ${expenses:.2f}
Net Balance: ${total:.2f}

Spending by Category:
"""
        for cat, amount in category_totals.items():
            summary += f"{cat}: ${amount:.2f}\n"

        return summary


manager = SimpleFinanceManager()


# -----------------------------------
# Functions
# -----------------------------------
def add_transaction_via_gradio(description, amount, category, date):
    if not description:
        return "Description is required."

    manager.add_transaction(description, amount, category, date)
    return f"Added transaction: {description} (${amount})"


def load_transactions_from_csv(csv_file):
    if csv_file is None:
        return "No file uploaded."

    try:
        df = pd.read_csv(csv_file.name)

        required_columns = ["description", "amount", "category", "date"]

        if not all(col in df.columns for col in required_columns):
            return f"CSV must include columns: {required_columns}"

        for _, row in df.iterrows():
            manager.add_transaction(
                row["description"],
                row["amount"],
                row["category"],
                row["date"]
            )

        return f"Successfully loaded {len(df)} transactions from CSV."

    except Exception as e:
        return f"Error: {str(e)}"


def get_analysis():
    return manager.get_summary()


# -----------------------------------
# Enhanced Gradio Interface
# -----------------------------------
with gr.Blocks(title="Smart Finance App v2") as enhanced_demo:

    gr.Markdown("# 💰 Smart Finance App v2")
    gr.Markdown("Track spending manually or import bank transactions via CSV.")

    # -------------------------------
    # Tab 1: Manual Transactions
    # -------------------------------
    with gr.Tab("Add Transactions"):
        gr.Markdown("## Add a Manual Transaction")

        desc_input = gr.Textbox(
            label="Description",
            placeholder="e.g. Grocery Shopping"
        )

        amount_input = gr.Number(
            label="Amount",
            value=0.0,
            info="Negative for expenses, positive for income"
        )

        category_input = gr.Dropdown(
            label="Category",
            choices=[
                "Food",
                "Transport",
                "Entertainment",
                "Bills",
                "Income",
                "Other"
            ]
        )

        date_input = gr.Textbox(
            label="Date",
            placeholder="YYYY-MM-DD"
        )

        add_button = gr.Button("Add Transaction", variant="primary")
        add_status = gr.Textbox(label="Status", interactive=False)

        add_button.click(
            fn=add_transaction_via_gradio,
            inputs=[desc_input, amount_input, category_input, date_input],
            outputs=add_status
        )

    # -------------------------------
    # Tab 2: CSV Upload
    # -------------------------------
    with gr.Tab("Load from CSV"):
        gr.Markdown("## Import Transactions from CSV")

        gr.Markdown("""
Upload a CSV file with columns:

- description
- amount
- category
- date
""")

        csv_upload = gr.File(
            label="Upload CSV File",
            file_types=[".csv"]
        )

        upload_button = gr.Button("Load CSV", variant="primary")
        upload_status = gr.Textbox(label="Upload Status", interactive=False)

        upload_button.click(
            fn=load_transactions_from_csv,
            inputs=csv_upload,
            outputs=upload_status
        )

    # -------------------------------
    # Tab 3: Analysis
    # -------------------------------
    with gr.Tab("Analysis"):
        gr.Markdown("## Spending Analysis")

        summary_output = gr.Textbox(
            label="Financial Summary",
            lines=15,
            interactive=False
        )

        refresh_button = gr.Button("Refresh Analysis")

        refresh_button.click(
            fn=get_analysis,
            outputs=summary_output
        )


# Launch app
enhanced_demo.launch(debug=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://095a44348d0bcbadd5.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://095a44348d0bcbadd5.gradio.live





---

## Part 7: Problem-Solving Analysis

### Task 7.1: Object Collaboration Mapping

Draw or describe how the different objects in your system work together:

**Gradio Objects:** (What Gradio objects did you use?)

**Your Custom Objects:** (Transaction, SimpleFinanceManager)

**Data Flow:** (How does information move between objects?)

### Task 7.2: Business Problem Solved

**1. Integration Problem:** How did your solution handle both manual entry AND CSV loading using the same objects?

**2. User Experience:** How do objects make the interface more reliable and user-friendly?

**3. Scalability:** How would your object-oriented design handle more features (budgets, categories, reports)?

---

## Part 8: AI-Assisted Enhancement

### Task 8.1: Feature Expansion

Ask AI to help you add one more feature to your finance app:

> "I want to add [CHOOSE: budget tracking / expense categories analysis / monthly reports / spending alerts] to my finance app. How would I enhance my existing transaction and manager systems to support this business feature while keeping everything organized and easy to maintain?"

```python
# AI's suggested enhancement:

```

### Task 8.2: Implementation and Testing

Implement the AI's suggestion and test it:

```python
# Your enhanced classes:

```

```python
# Test the new feature:

```

In [ ]:
# Adding budget tracking


import gradio as gr
import pandas as pd


# -----------------------------------
# Transaction Object
# -----------------------------------
class Transaction:
    def __init__(self, description, amount, category, date):
        self.description = description
        self.amount = float(amount)
        self.category = category
        self.date = date


# -----------------------------------
# Budget Manager
# -----------------------------------
class BudgetManager:
    def __init__(self):
        self.budgets = {}

    def set_budget(self, category, amount):
        self.budgets[category] = float(amount)

    def get_budget(self, category):
        return self.budgets.get(category, 0)

    def check_budget_status(self, transactions):
        spending = {}

        # Calculate spending by category
        for t in transactions:
            if t.amount < 0:  # expenses only
                spending[t.category] = spending.get(t.category, 0) + abs(t.amount)

        results = []

        for category, budget in self.budgets.items():
            spent = spending.get(category, 0)
            remaining = budget - spent

            if remaining < 0:
                status = f"⚠️ OVER budget by ${abs(remaining):.2f}"
            else:
                status = f"✅ ${remaining:.2f} remaining"

            results.append(
                f"{category}: Budget ${budget:.2f} | Spent ${spent:.2f} | {status}"
            )

        return "\n".join(results) if results else "No budgets set."


# -----------------------------------
# Finance Manager
# -----------------------------------
class FinanceManager:
    def __init__(self):
        self.transactions = []
        self.budget_manager = BudgetManager()

    def add_transaction(self, description, amount, category, date):
        transaction = Transaction(description, amount, category, date)
        self.transactions.append(transaction)

    def load_csv(self, csv_file):
        df = pd.read_csv(csv_file.name)

        for _, row in df.iterrows():
            self.add_transaction(
                row["description"],
                row["amount"],
                row["category"],
                row["date"]
            )

        return f"Loaded {len(df)} transactions."

    def get_summary(self):
        if not self.transactions:
            return "No transactions loaded."

        income = sum(t.amount for t in self.transactions if t.amount > 0)
        expenses = sum(t.amount for t in self.transactions if t.amount < 0)
        balance = income + expenses

        return f"""
Income: ${income:.2f}
Expenses: ${expenses:.2f}
Balance: ${balance:.2f}
"""

    def get_budget_report(self):
        return self.budget_manager.check_budget_status(self.transactions)


manager = FinanceManager()


# -----------------------------------
# Functions for Gradio
# -----------------------------------
def add_transaction_ui(description, amount, category, date):
    manager.add_transaction(description, amount, category, date)
    return "Transaction added successfully."


def upload_csv_ui(csv_file):
    return manager.load_csv(csv_file)


def set_budget_ui(category, amount):
    manager.budget_manager.set_budget(category, amount)
    return f"Budget set: {category} = ${amount}"


def refresh_summary():
    return manager.get_summary()


def refresh_budget_report():
    return manager.get_budget_report()


# -----------------------------------
# Gradio Interface
# -----------------------------------
with gr.Blocks(title="Smart Finance App v3") as demo:

    gr.Markdown("# 💰 Smart Finance App v3")
    gr.Markdown("Now with budget tracking.")

    # -------------------------------
    # Transactions Tab
    # -------------------------------
    with gr.Tab("Transactions"):
        desc = gr.Textbox(label="Description")
        amount = gr.Number(label="Amount")
        category = gr.Dropdown(
            choices=["Food", "Transport", "Entertainment", "Bills", "Income", "Other"],
            label="Category"
        )
        date = gr.Textbox(label="Date (YYYY-MM-DD)")

        add_btn = gr.Button("Add Transaction")
        add_output = gr.Textbox(label="Status")

        add_btn.click(
            fn=add_transaction_ui,
            inputs=[desc, amount, category, date],
            outputs=add_output
        )

    # -------------------------------
    # CSV Upload Tab
    # -------------------------------
    with gr.Tab("Load CSV"):
        csv_file = gr.File(file_types=[".csv"])
        upload_btn = gr.Button("Upload CSV")
        upload_output = gr.Textbox(label="Upload Status")

        upload_btn.click(
            fn=upload_csv_ui,
            inputs=csv_file,
            outputs=upload_output
        )

    # -------------------------------
    # Budget Tab
    # -------------------------------
    with gr.Tab("Budgets"):
        budget_category = gr.Dropdown(
            choices=["Food", "Transport", "Entertainment", "Bills", "Other"],
            label="Budget Category"
        )
        budget_amount = gr.Number(label="Monthly Budget")

        budget_btn = gr.Button("Set Budget")
        budget_output = gr.Textbox(label="Budget Status")

        budget_btn.click(
            fn=set_budget_ui,
            inputs=[budget_category, budget_amount],
            outputs=budget_output
        )

    # -------------------------------
    # Analysis Tab
    # -------------------------------
    with gr.Tab("Analysis"):
        summary_box = gr.Textbox(label="Summary", lines=8)
        budget_report_box = gr.Textbox(label="Budget Report", lines=10)

        summary_btn = gr.Button("Refresh Summary")
        budget_report_btn = gr.Button("Check Budgets")

        summary_btn.click(
            fn=refresh_summary,
            outputs=summary_box
        )

        budget_report_btn.click(
            fn=refresh_budget_report,
            outputs=budget_report_box
        )


demo.launch(debug=True)



---

## Reflection Questions

### Task 9.1: OOP Problem-Solving Insights

**1. Object Collaboration:** How did using multiple types of objects (Gradio objects + your custom objects) solve complex problems?

**2. Separation of Concerns:** How did keeping business logic (Transaction, Manager) separate from interface logic (Gradio) help?

**3. Real-World Application:** How is this pattern used in apps you use daily?

### Task 9.2: AI Development Partnership

**1. AI as Design Partner:** How did AI help you explore solutions you wouldn't have thought of?

**2. Human Oversight:** Where did you need to guide, correct, or enhance AI suggestions?

**3. Problem-Solving Process:** How did AI change your approach to building software?

---

## Extension Challenges (Optional)

### Challenge 1: Smart Chatbot Integration
Add a simple chatbot that can answer questions about spending using your objects.

### Challenge 2: Data Visualization  
Add charts to visualize spending patterns using your transaction data.

### Challenge 3: Export Functionality
Add the ability to export transaction data back to CSV format.

---

## Key Takeaways

You've now built a working finance application that demonstrates:

- **Object Collaboration:** How different types of objects work together
- **Separation of Concerns:** UI objects vs business logic objects
- **Scalable Design:** How OOP makes adding features easier  
- **Real-World Integration:** How to connect file data, manual input, and user interfaces
- **AI-Assisted Development:** How to effectively partner with AI in building applications

Most importantly: You've seen how object-oriented programming solves real business problems by organising code the same way businesses organise their operations.